In [34]:
import torch
import torch.nn.functional as F
from transformers import PreTrainedTokenizer, PreTrainedModel
import checkpoint
import rich
import pprint
import train_util

In [4]:
model, tokenizer = checkpoint.get_model_and_tokenizer(checkpoint.OLMO, "cpu")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 179/179 [00:09<00:00, 18.71it/s]


In [14]:
prompt_strs = ["this is the first prompt:", "second promp:"]
output_strs = ["hello world nice prompt", "gah"]

full_toks = [tokenizer.encode(prompt_str) + tokenizer.encode(output_str) for prompt_str, output_str in zip(prompt_strs, output_strs)]

batch_size = len(full_toks)

tokenizer.pad(encoded_inputs={"input_ids": full_toks}, padding="longest", padding_side="left", return_tensors="pt", atte)

{'input_ids': tensor([[   576,    374,    279,   1176,  10137,     25,  15339,   1917,   6555,
          10137],
        [100277, 100277, 100277, 100277,   5686,   2773,     79,     25,     70,
           1494]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1, 1, 1]])}

In [22]:
res = tokenizer_prompt_and_output(prompt_strs, output_strs, tokenizer)

In [39]:
input_ids = res["input_ids"]
labels = res["labels"]

In [49]:
def get_entropy(logprobs, dim):
    return - torch.sum(logprobs * torch.exp(logprobs), dim=dim)


def get_response_log_probs(
        model: PreTrainedModel, 
        input_ids: torch.Tensor, 
        labels: torch.Tensor, 
        return_token_entropy: bool = False,
) -> dict[str, torch.Tensor]:


    model.eval()
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits  # Shape: [batch_size, sequence_length, vocab_size]

    log_probs = F.log_softmax(logits, dim=-1)
    target_log_probs = torch.gather(log_probs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
    out_dict = {"log_probs": target_log_probs}

    if return_token_entropy: 
        token_entropy = get_entropy(log_probs, dim=-1)
        out_dict["token_entropy"] = token_entropy

    return out_dict


In [50]:
get_response_log_probs(model, input_ids, labels, return_token_entropy = True)

{'log_probs': tensor([[ -1.5156,  -1.8281,  -2.6250, -10.8750,  -6.1562, -12.1875,  -1.5547,
           -9.0000,  -2.5000],
         [-19.5000, -19.5000, -19.5000, -16.7500, -12.0000,  -6.0938,  -5.0938,
          -11.5000,  -7.3750]], dtype=torch.bfloat16),
 'token_entropy': tensor([[6.0312, 4.8750, 6.5000, 4.9688, 4.1562, 6.4688, 4.8125, 5.2188, 4.9688],
         [7.6562, 7.6562, 7.6562, 7.5938, 7.2500, 5.3438, 7.7500, 6.6562, 7.3125]],
        dtype=torch.bfloat16)}

In [51]:
model.eval()
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits  # Shape: [batch_size, sequence_length, vocab_size]

log_probs = F.log_softmax(logits, dim=-1)
target_log_probs = torch.gather(log_probs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
out_dict = {"log_probs": target_log_probs}

token_entropy = get_entropy(log_probs, dim=-1)
out_dict["token_entropy"] = token_entropy

In [52]:
log_probs * torch.exp(log_probs)

tensor([[[-6.0730e-03, -5.2490e-02, -3.7231e-03,  ..., -2.5611e-08,
          -2.5611e-08, -2.5611e-08],
         [-3.6240e-04, -7.8735e-03, -6.9618e-05,  ..., -1.7928e-08,
          -1.7928e-08, -1.7928e-08],
         [-2.2054e-05, -1.0395e-04, -2.7120e-06,  ..., -4.6566e-08,
          -4.6566e-08, -4.6566e-08],
         ...,
         [-1.0107e-01, -6.0730e-03, -2.9144e-03,  ..., -5.4424e-09,
          -5.4424e-09, -5.4424e-09],
         [-1.9531e-01, -5.4932e-02, -6.7139e-03,  ..., -4.8312e-09,
          -4.8312e-09, -4.8312e-09],
         [-5.6396e-02, -3.9368e-03, -3.6955e-05,  ..., -3.8126e-09,
          -3.8126e-09, -3.8126e-09]],

        [[-2.1515e-03, -2.1744e-04, -9.2387e-06,  ..., -6.6590e-08,
          -6.6590e-08, -6.6590e-08],
         [-2.1515e-03, -2.1744e-04, -9.2387e-06,  ..., -6.6590e-08,
          -6.6590e-08, -6.6590e-08],
         [-2.1515e-03, -2.1744e-04, -9.2387e-06,  ..., -6.6590e-08,
          -6.6590e-08, -6.6590e-08],
         ...,
         [-8.3008e-03, -6